# Antardrishti — where the missing water is

**Question this notebook answers:** Sentinel-1 VV at −17 dB scores recall **0.511** on Sen1Floods11. Half the water in the labels is not detected. Is that fixable with VH, or with a VV+VH fusion rule?

**Why the threshold sweep already ruled out the easy answer.** `evaluation/sen1floods11_results.json` shows precision collapsing faster than recall rises as the threshold moves up: at −14 dB recall reaches 0.650 but precision falls to 0.478. So the missed pixels are mostly *not* sitting just above the line.

**The hypothesis.** Flooded vegetation is invisible to a darkness rule. The pulse hits the water, bounces off trunks and stems, and returns to the sensor — *double bounce*. That is **bright**, not dark. Double bounce preserves polarisation, so it lifts VV far more than VH, leaving a wide VV−VH gap. A dry canopy scatters inside the foliage instead, which depolarises, narrowing the gap. So VV−VH should separate the two.

**What would count as an answer.** Not "the fusion rule scored higher". The diagnostic in section 5 bins every missed water pixel by brightness and reports what share carries the double-bounce signature. That share *is* the answer to "how much of the missing recall is flooded vegetation?"

**The caveat that outranks everything below.** Sen1Floods11 labels were drawn from Sentinel-2 NDVI/MNDWI and hand corrected using optical imagery, DEM, land cover and slope. None of those reveal water under a canopy. If flooded vegetation is not in the labels, detecting it correctly scores as a **false positive**, and no fusion rule can win no matter how good the physics. Section 6 states this next to whatever the numbers say.

**Runs on your own machine.** No GPU, no cloud account — this is all numpy. Budget about 2 GB of download (once), 2 GB of free RAM, and a couple of minutes of CPU.

Costs nothing extra to download: the `S1Hand` chips notebook 01 already used carry **both** polarisations. Band 1 is VV, band 2 is VH.

In [ ]:
# One-time install. rasterio ships Windows wheels, so there is no compiler step.
!pip install rasterio numpy


## 1. Data

Same 446 hand-labelled chips as notebook 01, so the numbers here are directly comparable to `sen1floods11_results.json`. List before assuming — these paths have changed between dataset versions.

## 1. Data

Same 446 hand-labelled chips as notebook 01, so these numbers are directly
comparable to `sen1floods11_results.json`.

The original lives in a Google Cloud bucket and needs the Cloud SDK. There is a
Hugging Face mirror that is a single file and needs no account, which is much
easier on Windows. **Run these three commands in your terminal once:**

```
pip install huggingface_hub
hf download blumenstiel/Sen1Floods11 sen1floods11_v1_1.tar.gz --repo-type dataset --local-dir .
tar -xf sen1floods11_v1_1.tar.gz
```

1.77 GB, and you only pay it once. This is the labelled subset only — the
weakly-labelled S1 data is far larger and is not needed to measure accuracy.

Already have the chips from notebook 01? Point `LOCAL` at that folder and skip
the download entirely.


In [ ]:
# Wherever you extracted the archive. Windows paths need the r"" prefix.
LOCAL = r"C:\sen1floods11"

from pathlib import Path

root = Path(LOCAL)
if not root.exists():
    raise SystemExit(f"{root} does not exist - see the download commands above")

# The Google Cloud layout and the Hugging Face layout put these in different
# folders, so search instead of assuming. Both name the files the same way,
# which is the part that actually matters.
s1_files = sorted(root.rglob("*_S1Hand.tif"))
label_files = sorted(root.rglob("*_LabelHand.tif"))

print(f"S1 chips: {len(s1_files)}")
print(f"labels  : {len(label_files)}")
if s1_files:
    print(f"found under: {s1_files[0].parent}")
else:
    print("\nNothing found. Check the path - the files may be one level deeper.")


## 2. Our own code

`thresholds.py`, `metrics.py` and `diagnostics.py` are already in your repo under
`application/antardrishti/evaluation/`. Nothing to upload — the next cell just
puts that folder on the import path.

The fusion rules and the diagnostic are unit-tested in
`backend/tests/test_polarisation.py`, so what runs here is what passed there.


In [ ]:
import sys
from pathlib import Path

# This notebook lives in backend/notebooks/, so evaluation/ is two levels up.
EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
if not EVALUATION.exists():
    raise SystemExit(f"{EVALUATION} not found - start Jupyter from backend/notebooks/")

sys.path.insert(0, str(EVALUATION))
print(f"importing from {EVALUATION}")


In [ ]:
import json

import numpy as np
import rasterio

from metrics import evaluate, aggregate, format_table
from diagnostics import ErrorProfile, interpret
import thresholds as T

# Some tiles carry a sentinel value rather than NaN for missing data. Nothing
# real is darker than this, so anything below it is nodata whatever it claims.
NODATA_FLOOR_DB = -50.0

def chip_pairs():
    """Match each S1 chip to its label by the shared prefix."""
    labels = {p.name.replace("_LabelHand.tif", ""): p for p in label_files}
    pairs = []
    for path in s1_files:
        key = path.name.replace("_S1Hand.tif", "")
        if key in labels:
            pairs.append((key, path, labels[key]))
    return pairs

def load_dual(s1_path, label_path):
    """Band 1 = VV, band 2 = VH, both in dB. Labels: -1 nodata, 0 land, 1 water."""
    with rasterio.open(s1_path) as src:
        vv = src.read(1).astype(np.float64)
        vh = src.read(2).astype(np.float64)
    with rasterio.open(label_path) as src:
        truth = src.read(1).astype(np.int8)

    vv[vv < NODATA_FLOOR_DB] = np.nan
    vh[vh < NODATA_FLOOR_DB] = np.nan
    return vv, vh, truth

pairs = chip_pairs()
print(f"{len(pairs)} matched chip/label pairs")
if not pairs:
    raise SystemExit("no pairs matched - check the filenames listed above")

vv, vh, truth = load_dual(*pairs[0][1:])
print(f"VV {np.nanmin(vv):.1f} to {np.nanmax(vv):.1f} dB")
print(f"VH {np.nanmin(vh):.1f} to {np.nanmax(vh):.1f} dB")
print("label values", np.unique(truth))


## 3. Check the physics before trusting any rule built on it

Everything below assumes water is darker than land in both bands, and that the VV−VH gap carries information. If that is not visible in the labelled pixels, stop here — the rules are built on sand.

Pooled over every chip, so one unusual scene cannot set the expectation.

In [ ]:
water_vv, water_vh, land_vv, land_vh = [], [], [], []

for key, s1_path, label_path in pairs:
    try:
        vv, vh, truth = load_dual(s1_path, label_path)
    except Exception:
        continue

    ok = (truth != -1) & np.isfinite(vv) & np.isfinite(vh)
    if not ok.any():
        continue

    # Subsample: the medians are stable long before all 100M pixels are in.
    w, l = ok & (truth == 1), ok & (truth == 0)
    water_vv.append(vv[w][::37]); water_vh.append(vh[w][::37])
    land_vv.append(vv[l][::37]);  land_vh.append(vh[l][::37])

water_vv = np.concatenate(water_vv); water_vh = np.concatenate(water_vh)
land_vv = np.concatenate(land_vv);   land_vh = np.concatenate(land_vh)

print(f"{'':<10}{'VV':>9}{'VH':>9}{'VV-VH':>9}")
print("-" * 37)
for name, a, b in (("water", water_vv, water_vh), ("land", land_vv, land_vh)):
    print(f"{name:<10}{np.median(a):>9.1f}{np.median(b):>9.1f}{np.median(a - b):>9.1f}")

separation = np.median(land_vv) - np.median(water_vv)
print(f"\nVV separation: {separation:.1f} dB")
print(f"VH separation: {np.median(land_vh) - np.median(water_vh):.1f} dB")
print("\nIf VH separates water from land as well as VV does, a VH rule is worth")
print("having. If it separates them BETTER, that alone is the finding.")

## 4. Find the VH threshold, then score every rule

The −22 dB default in `thresholds.py` is a starting point from the physics, not a measured value. Sweep it the same way notebook 01 swept VV, then rebuild the fusion rules around whatever wins.

In [ ]:
# Holding every chip in memory costs roughly 2 GB. Short on RAM? Set LIMIT to
# 100 for a fast first pass - the conclusions hold, only the decimals move.
LIMIT = None

loaded = []
for key, s1_path, label_path in (pairs[:LIMIT] if LIMIT else pairs):
    try:
        vv, vh, truth = load_dual(s1_path, label_path)
    except Exception as exc:
        print(f"skip {key}: {exc}")
        continue
    if (truth != -1).sum():
        loaded.append((vv, vh, truth))

print(f"{len(loaded)} chips held in memory")

vh_sweep = {}
for db in np.arange(-30.0, -13.0, 1.0):
    method = T.dual_fixed_vh(float(db))
    vh_sweep[float(db)] = aggregate(
        [evaluate(method(vv, vh)[0], truth) for vv, vh, truth in loaded]
    )

print(f"\n{'VH dB':>7}{'IoU':>8}{'F1':>8}{'prec':>8}{'recall':>8}")
print("-" * 39)
best_vh = None
for db, agg in vh_sweep.items():
    m = agg["micro"]
    print(f"{db:>7.0f}{m['iou']:>8.3f}{m['f1']:>8.3f}{m['precision']:>8.3f}{m['recall']:>8.3f}")
    if best_vh is None or m["iou"] > best_vh[1]:
        best_vh = (float(db), m["iou"])

BEST_VH_DB = best_vh[0]
print(f"\nBest VH threshold: {BEST_VH_DB:.0f} dB at IoU {best_vh[1]:.3f}")
print(f"VV, for comparison: -17 dB at IoU 0.441")

In [ ]:
# Sweep the double-bounce gap too, for the same reason.
gap_sweep = {}
for gap in np.arange(5.0, 15.0, 1.0):
    method = T.dual_open_water_or_double_bounce(-17.0, float(gap))
    gap_sweep[float(gap)] = aggregate(
        [evaluate(method(vv, vh)[0], truth) for vv, vh, truth in loaded]
    )

print(f"{'gap dB':>7}{'IoU':>8}{'prec':>8}{'recall':>8}")
print("-" * 31)
best_gap = None
for gap, agg in gap_sweep.items():
    m = agg["micro"]
    print(f"{gap:>7.0f}{m['iou']:>8.3f}{m['precision']:>8.3f}{m['recall']:>8.3f}")
    if best_gap is None or m["iou"] > best_gap[1]:
        best_gap = (float(gap), m["iou"])

BEST_GAP_DB = best_gap[0]
print(f"\nBest gap: {BEST_GAP_DB:.0f} dB at IoU {best_gap[1]:.3f}")
print("\nA gap threshold this rule likes at the very edge of the sweep range")
print("means the rule is drifting toward 'everything' or 'nothing' - check")
print("recall and precision above before believing the IoU.")

In [ ]:
TUNED = {
    "VV only -17 dB (ships today)": T.dual_fixed_vv(-17.0),
    f"VH only {BEST_VH_DB:.0f} dB": T.dual_fixed_vh(BEST_VH_DB),
    "VV or VH (union)": T.dual_union(-17.0, BEST_VH_DB),
    "VV and VH (intersection)": T.dual_intersection(-17.0, BEST_VH_DB),
    "mean(VV,VH)": T.dual_mean((-17.0 + BEST_VH_DB) / 2.0),
    f"water or double bounce ({BEST_GAP_DB:.0f} dB)":
        T.dual_open_water_or_double_bounce(-17.0, BEST_GAP_DB),
}

results = {}
for name, method in TUNED.items():
    results[name] = aggregate(
        [evaluate(method(vv, vh)[0], truth) for vv, vh, truth in loaded]
    )

print(format_table(results, "Sen1Floods11 — dual polarisation"))

baseline = results["VV only -17 dB (ships today)"]["micro"]
print(f"\n{'rule':<38}{'d IoU':>9}{'d recall':>10}{'d prec':>9}")
print("-" * 66)
for name, agg in results.items():
    m = agg["micro"]
    print(f"{name:<38}{m['iou'] - baseline['iou']:>+9.3f}"
          f"{m['recall'] - baseline['recall']:>+10.3f}"
          f"{m['precision'] - baseline['precision']:>+9.3f}")

print("\nA rule only earns its way into backend/detection/sar.py if IoU rises. Recall")
print("bought with more precision than it is worth is not an improvement.")

## 5. The diagnostic — what are the missed pixels actually made of?

This is the part that answers the question, regardless of whether any rule above won.

Every water pixel the shipping method misses gets binned:

| bin | meaning | response |
|---|---|---|
| **marginal** | within 3 dB of the threshold | calibration — the sweep already prices this |
| **moderate** | darker than land, well above the threshold | mixed pixels, shallow or sediment-laden water, wind |
| **bright** | above −10 dB | too bright to be open water at any threshold |

and of the bright ones, those with a VV−VH gap wide enough to mean the return kept its polarisation — **double bounce**, the flooded-vegetation signature.

In [ ]:
shipping = T.dual_fixed_vv(-17.0)
profile = ErrorProfile(threshold=-17.0, double_bounce_gap=BEST_GAP_DB)

for vv, vh, truth in loaded:
    profile.update(vv, vh, truth, shipping(vv, vh)[0])

summary = profile.summary()

print(f"{'outcome':<18}{'pixels':>13}{'VV med':>9}{'VV p90':>9}{'gap med':>9}")
print("-" * 58)
def db(value):
    # An outcome with no pixels has no median. Say so rather than crashing.
    return f"{value:>9.1f}" if value is not None else f"{'-':>9}"

for outcome, stats in summary["by_outcome"].items():
    print(f"{outcome:<18}{stats['pixels']:>13,}"
          f"{db(stats['vv_median_db'])}{db(stats['vv_p90_db'])}"
          f"{db(stats['vv_vh_gap_median_db'])}")

missed = summary["missed_water"]
print(f"\nMissed water: {missed['total_pixels']:,} pixels")
print(f"{'bin':<12}{'pixels':>13}{'share':>9}")
print("-" * 34)
for name, count in missed["bins"].items():
    print(f"{name:<12}{count:>13,}{missed['shares'][name]:>9.1%}")
print(f"\n  of which bright AND double-bounce: "
      f"{missed['bright_and_double_bounce_pixels']:,} "
      f"({missed['bright_and_double_bounce_share']:.1%} of all missed water)")

## 6. The reading

`interpret()` is deliberately conservative: it reports the shares, names the confound, and always states the label caveat. It does not declare the hypothesis proved.

In [ ]:
for line in interpret(summary):
    print(f"- {line}\n")

In [ ]:
payload = {
    "dataset": "Sen1Floods11 HandLabeled",
    "chips_scored": len(loaded),
    "question": "Is the 0.489 missing recall recoverable with VH or VV+VH fusion?",
    "baseline": {"method": "VV -17 dB", "iou": 0.441, "recall": 0.511},
    "tuned_thresholds": {"vh_db": BEST_VH_DB, "double_bounce_gap_db": BEST_GAP_DB},
    "vh_sweep": {str(db): agg["micro"] for db, agg in vh_sweep.items()},
    "gap_sweep": {str(gap): agg["micro"] for gap, agg in gap_sweep.items()},
    "methods": {
        name: {"micro": agg["micro"], "macro": agg["macro"]}
        for name, agg in results.items()
    },
    "missing_recall_diagnostic": summary,
    "interpretation": interpret(summary),
    "label_caveat": (
        "Sen1Floods11 labels were drawn from Sentinel-2 NDVI/MNDWI and hand "
        "corrected using optical imagery, DEM, land cover and slope. Water "
        "under a canopy is not visible in any of those, so flooded vegetation "
        "may be absent from the ground truth. Detecting it would then score as "
        "a false positive rather than as recovered recall."
    ),
}

out = EVALUATION / "polarisation_results.json"
with out.open("w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2)

print(f"written to {out}")

## What to do with the numbers

Decide from the table in section 4 and the shares in section 5, in that order.

**If a fusion rule beats IoU 0.441.** It earns its way into `backend/detection/sar.py`. Add the polarisation to `get_collection`, change `detect_water` to build the fused mask, and update the `validation` block in the `info` dict to the measured numbers — that block is read directly by the report generator, so a stale value there becomes a false claim in a user-facing answer. `evaluation/thresholds.py` and `backend/detection/sar.py` are two implementations of one method; change both or the evaluation stops measuring what ships.

**If nothing beats 0.441.** That is a result, not a failure, and a more interesting one than a small win. It means the missing half of the water is not recoverable from polarisation at all, and the honest line for Chapter 7 is that recall 0.511 is a property of single-date threshold detection rather than of the threshold. Report it and move on.

**If the bright + double-bounce share is large** (say above 20%), flooded vegetation really is a major part of the gap. But note what that implies: those pixels are bright, so the fix is not a better threshold but **change detection** — comparing backscatter against a dry baseline and looking for an *increase*. The backend already computes a pre-flood composite in comparison mode (`pre_start`/`pre_end` in `analysis.py`), so the machinery is half there.

**If the marginal share is large**, the sweep in notebook 01 already tells you the price. Nothing new to build.

**Whatever happens, check the double-bounce rule's precision before shipping it.** Built-up areas scatter exactly the same way. A rule that finds flooded forest and also paints every town as flooded is worse than the one that ships today. `backend/tests/test_polarisation.py` has a test asserting built-up triggers the rule, specifically so this cannot be forgotten.